In [18]:
import pandas as pd
import numpy as np
import faiss
import time
import re

from pathlib import Path
from sentence_transformers import SentenceTransformer

In [19]:
from pathlib import Path

dataset_path = Path(
    r"C:\Users\me\.cache\kagglehub\datasets\rmisra\news-category-dataset\versions\3"
)

for file in dataset_path.iterdir():
    print(file)

C:\Users\me\.cache\kagglehub\datasets\rmisra\news-category-dataset\versions\3\News_Category_Dataset_v3.json


In [20]:
df = pd.read_json(
    dataset_path / "News_Category_Dataset_v3.json",
    lines=True
)

print(df.shape)
print(df.head())

(209527, 6)
                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you 

In [23]:
article_text = (
    df["headline"].fillna("")
    + ". "
    + df["short_description"].fillna("")
)

print(article_text.iloc[0])

Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters. Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.


In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
test_embedding = model.encode(
    [article_text.iloc[0]],
    convert_to_numpy=True
)

print(test_embedding.shape)

(1, 384)


In [26]:
from pathlib import Path

embedding_dir = Path("embeddings_summary")
embedding_dir.mkdir(exist_ok=True)

print("Embedding folder:", embedding_dir)

Embedding folder: embeddings_summary


In [27]:
embedding_dir.mkdir(exist_ok=True)

In [28]:
import numpy as np
import time

chunk_size = 20_000

for start in range(0, len(df), chunk_size):

    end = min(start + chunk_size, len(df))

    output_file = embedding_dir / f"embeddings_{start}_{end}.npy"

    if output_file.exists():
        print(f"Skipping {start}:{end} - already exists")
        continue

    print(f"\nProcessing articles {start}:{end}")

    start_time = time.time()

    chunk_text = article_text.iloc[start:end].tolist()

    chunk_embeddings = model.encode(
        chunk_text,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    chunk_embeddings = np.asarray(
        chunk_embeddings,
        dtype="float32"
    )

    np.save(output_file, chunk_embeddings)

    elapsed = time.time() - start_time

    print(
        f"Saved {chunk_embeddings.shape} "
        f"to {output_file} "
        f"in {elapsed:.2f} seconds"
    )

Skipping 0:20000 - already exists
Skipping 20000:40000 - already exists
Skipping 40000:60000 - already exists
Skipping 60000:80000 - already exists
Skipping 80000:100000 - already exists
Skipping 100000:120000 - already exists
Skipping 120000:140000 - already exists
Skipping 140000:160000 - already exists
Skipping 160000:180000 - already exists
Skipping 180000:200000 - already exists
Skipping 200000:209527 - already exists


In [29]:
embedding_files = sorted(
    embedding_dir.glob("*.npy")
)

for file in embedding_files:
    print(file.name)

embeddings_0_20000.npy
embeddings_100000_120000.npy
embeddings_120000_140000.npy
embeddings_140000_160000.npy
embeddings_160000_180000.npy
embeddings_180000_200000.npy
embeddings_200000_209526.npy
embeddings_200000_209527.npy
embeddings_20000_40000.npy
embeddings_40000_60000.npy
embeddings_60000_80000.npy
embeddings_80000_100000.npy


In [17]:
import numpy as np
from pathlib import Path

embedding_dir = Path("embeddings_summary")

embedding_files = sorted(
    embedding_dir.glob("*.npy")
)

embedding_files = [
    f for f in embedding_files
    if f.name != "embeddings_200000_209527.npy"
]

for file in embedding_files:
    emb = np.load(file, mmap_mode="r")
    print(f"{file.name}: {emb.shape}")

embeddings_0_20000.npy: (20000, 384)
embeddings_100000_120000.npy: (20000, 384)
embeddings_120000_140000.npy: (20000, 384)
embeddings_140000_160000.npy: (20000, 384)
embeddings_160000_180000.npy: (20000, 384)
embeddings_180000_200000.npy: (20000, 384)
embeddings_200000_209526.npy: (9526, 384)
embeddings_20000_40000.npy: (20000, 384)
embeddings_40000_60000.npy: (20000, 384)
embeddings_60000_80000.npy: (20000, 384)
embeddings_80000_100000.npy: (20000, 384)


In [31]:
embedding_files = sorted(
    embedding_files,
    key=lambda x: int(x.stem.split("_")[1])
)

for file in embedding_files:
    print(file.name)

embeddings_0_20000.npy
embeddings_20000_40000.npy
embeddings_40000_60000.npy
embeddings_60000_80000.npy
embeddings_80000_100000.npy
embeddings_100000_120000.npy
embeddings_120000_140000.npy
embeddings_140000_160000.npy
embeddings_160000_180000.npy
embeddings_180000_200000.npy
embeddings_200000_209526.npy
embeddings_200000_209527.npy


In [35]:
embedding_files = sorted(
    embedding_dir.glob("*.npy"),
    key=lambda x: int(x.stem.split("_")[1])
)

# Remove the incorrect chunk
embedding_files = [
    file for file in embedding_files
    if file.name != "embeddings_200000_209527.npy"
]

for file in embedding_files:
    print(file.name)

embeddings_0_20000.npy
embeddings_20000_40000.npy
embeddings_40000_60000.npy
embeddings_60000_80000.npy
embeddings_80000_100000.npy
embeddings_100000_120000.npy
embeddings_120000_140000.npy
embeddings_140000_160000.npy
embeddings_160000_180000.npy
embeddings_180000_200000.npy
embeddings_200000_209526.npy


In [36]:
embeddings = np.vstack([
    np.load(file)
    for file in embedding_files
])

print("Final shape:", embeddings.shape)

Final shape: (209526, 384)


In [37]:
from sklearn.preprocessing import normalize

embeddings = normalize(
    embeddings,
    axis=1
).astype("float32")

print("Shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

Shape: (209526, 384)
Dtype: float32


In [38]:
import numpy as np

norms = np.linalg.norm(embeddings, axis=1)

print("First 5 norms:", norms[:5])
print("Minimum norm:", norms.min())
print("Maximum norm:", norms.max())

First 5 norms: [1.         1.         1.         0.99999994 1.        ]
Minimum norm: 0.99999976
Maximum norm: 1.0000002


In [39]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

print("Dimension:", dimension)
print("Vectors before adding:", index.ntotal)

Dimension: 384
Vectors before adding: 0


In [40]:
index.add(embeddings)

print("Vectors after adding:", index.ntotal)


Vectors after adding: 209526


In [41]:
query_vector = embeddings[0].reshape(1, -1)
k = 5
similarity_scores, article_indices = index.search(query_vector, k)

print("\nSimilarity scores:")
print(similarity_scores)

print("\nArticle indices:")
print(article_indices)


Similarity scores:
[[0.99999994 0.5899376  0.5604471  0.55515826 0.548419  ]]

Article indices:
[[   0 1497 3312 5188 1543]]


In [42]:
print("Articles:", len(df))
print("Embeddings:", len(embeddings))

print("Same number:", len(df) == len(embeddings))

Articles: 209527
Embeddings: 209526
Same number: False


In [43]:
print(df.tail(5)[["headline", "category"]])

                                                 headline category
209522  RIM CEO Thorsten Heins' 'Significant' Plans Fo...     TECH
209523  Maria Sharapova Stunned By Victoria Azarenka I...   SPORTS
209524  Giants Over Patriots, Jets Over Colts Among  M...   SPORTS
209525  Aldon Smith Arrested: 49ers Linebacker Busted ...   SPORTS
209526  Dwight Howard Rips Teammates After Magic Loss ...   SPORTS


In [44]:
print(df.index[-5:])

RangeIndex(start=209522, stop=209527, step=1)


In [45]:
print("Last embedding index:", len(embeddings) - 1)
print("Last dataframe position:", len(df) - 1)

Last embedding index: 209525
Last dataframe position: 209526


In [47]:
df = df.iloc[:209526].copy()

print("Articles:", len(df))
print("Embeddings:", len(embeddings))
print("Same number:", len(df) == len(embeddings))

Articles: 209526
Embeddings: 209526
Same number: True


In [53]:
def recommend_articles(query_index, k=5):
    
    query_article = df.iloc[query_index]
    query_vector = embeddings[query_index].reshape(1, -1)
    
    scores, indices = index.search(query_vector, k + 1)
    
    print("=" * 80)
    print("QUERY ARTICLE")
    print("=" * 80)
    
    print(f"Index:    {query_index}")
    print(f"Headline: {query_article['headline']}")
    print(f"Category: {query_article['category']}")
    
    print("\n" + "=" * 80)
    print("RECOMMENDED ARTICLES")
    print("=" * 80)
    
    recommendation_number = 1
    
    for score, idx in zip(scores[0], indices[0]):
        
        # Don't recommend the article itself
        if idx == query_index:
            continue
        
        article = df.iloc[idx]
        
        print(f"\nRecommendation {recommendation_number}")
        print("-" * 80)
        print(f"Index:      {idx}")
        print(f"Similarity: {score:.4f}")
        print(f"Category:   {article['category']}")
        print(f"Headline:   {article['headline']}")
        
        recommendation_number += 1
        
        if recommendation_number > k:
            break

In [55]:
recommend_articles(63070, k=5)

QUERY ARTICLE
Index:    63070
Headline: Amber Rose Encourages Iggy Azalea To 'Date A Bunch Of Hot Guys' To Get Over Nick Young
Category: ENTERTAINMENT

RECOMMENDED ARTICLES

Recommendation 1
--------------------------------------------------------------------------------
Index:      114103
Similarity: 0.6371
Category:   ENTERTAINMENT
Headline:   Iggy Azalea & Nick Young Remember Their First Encounter Differently

Recommendation 2
--------------------------------------------------------------------------------
Index:      71219
Similarity: 0.6163
Category:   ENTERTAINMENT
Headline:   Iggy Azalea Speaks Out About Nick Young Cheating Rumors

Recommendation 3
--------------------------------------------------------------------------------
Index:      70405
Similarity: 0.6008
Category:   ENTERTAINMENT
Headline:   Iggy Azalea Says Nick Young Will Have 'Half A Penis' If He Cheats On Her

Recommendation 4
--------------------------------------------------------------------------------
Index:  

In [56]:
import faiss

faiss.write_index(
    index,
    "news_rec_embeddings.faiss"
)

print("FAISS index saved successfully.")

FAISS index saved successfully.


In [57]:
from pathlib import Path

index_path = Path("news_rec_embeddings.faiss")

print("Exists:", index_path.exists())
print("Size:", index_path.stat().st_size / (1024 ** 2), "MB")

Exists: True
Size: 306.92289447784424 MB


In [58]:
df = df.iloc[:209526].copy()

print("Articles:", len(df))
print("Embeddings:", len(embeddings))

Articles: 209526
Embeddings: 209526


In [59]:
df.to_parquet(
    "news_articles.parquet",
    index=False
)

print("Article metadata saved successfully.")

Article metadata saved successfully.


In [62]:
print(model)
print(model.get_sentence_embedding_dimension())

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)
384


C:\Users\me\AppData\Local\Temp\ipykernel_6476\2868504806.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(model.get_sentence_embedding_dimension())


In [63]:
print("Model path:", model.model_card_data)

Model path: tags:
- sentence-transformers
- sentence-similarity
- feature-extraction
- dense
base_model: sentence-transformers/all-MiniLM-L6-v2
pipeline_tag: sentence-similarity
library_name: sentence-transformers


In [64]:
print(model[0].auto_model.name_or_path)

sentence-transformers/all-MiniLM-L6-v2


In [65]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully.")
print("Embedding dimension:", embedding_model.get_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully.
Embedding dimension: 384


In [66]:
test_text = "BlackBerry smartphone business"

test_embedding = embedding_model.encode(
    test_text,
    convert_to_numpy=True
)

print("Shape:", test_embedding.shape)
print("Dtype:", test_embedding.dtype)

Shape: (384,)
Dtype: float32


In [67]:
import numpy as np

test_embedding = test_embedding.astype("float32")

test_embedding = test_embedding / np.linalg.norm(
    test_embedding
)

print("Norm:", np.linalg.norm(test_embedding))

Norm: 1.0
